In [2]:
import os
import cv2
import numpy as np
import pandas as pd

def create_msi_dataset(base_folder, output_folder, num_subjects=None, camera_angle=None):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    msi_folder = os.path.join(output_folder, 'msi_images')
    if not os.path.exists(msi_folder):
        os.makedirs(msi_folder)
    
    metadata = []
    counter = 0
    
    subject_folders = [f for f in os.listdir(base_folder) if f.isdigit()]
    if num_subjects is not None:
        subject_folders = subject_folders[:num_subjects]
    
    for subject_folder in subject_folders:
        counter += 1
        subject_path = os.path.join(base_folder, subject_folder, subject_folder)
        if not os.path.isdir(subject_path):
            print(f"No inner subject folder found for: {subject_folder}")
            continue
        
        for nm_folder in ['nm-01', 'nm-02', 'nm-03', 'nm-04', 'nm-05', 'nm-06']:
            nm_folder_path = os.path.join(subject_path, nm_folder)
            if not os.path.exists(nm_folder_path):
                print(f"No {nm_folder} folder found for subject: {subject_folder}")
                continue

            for cam_folder in camera_angle:
                cam_path = os.path.join(nm_folder_path, cam_folder)
                if not os.path.isdir(cam_path):
                    continue
                
                image_files = sorted([f for f in os.listdir(cam_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                num_images = len(image_files)
                
                if num_images == 0:
                    print(f"No images found in {cam_path}")
                    continue
                
                sequence_images = []
                for file in image_files:
                    image_path = os.path.join(cam_path, file)
                    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                    if image is not None and image.size > 0:
                        sequence_images.append(image)
                    else:
                        print(f"Failed to load image or empty image: {image_path}")
                
                if sequence_images:
                    msi = motion_silhouette_image(sequence_images)
                    if msi is not None and msi.size > 0:
                        if msi.shape != (240, 240):
                            try:
                                msi = cv2.resize(msi, (240, 240))
                            except cv2.error:
                                print(f"Failed to resize MSI from {msi.shape} to (240, 240) for {cam_path}")
                                continue
                        
                        msi_filename = f"{subject_folder}_{nm_folder}_{cam_folder}_msi.png"
                        msi_path = os.path.join(msi_folder, msi_filename)
                        cv2.imwrite(msi_path, msi)
                        
                        metadata.append({
                            'image_filename': msi_filename,
                            'label': int(subject_folder),
                            'subject_id': int(subject_folder),
                            'cam_angle': int(cam_folder),
                            'original_image_path': cam_path
                        })
                    else:
                        print(f"Failed to create valid MSI for {cam_path}")
                
        print(f"{counter}: Finished processing subject: {subject_folder}")
    
    if metadata:
        metadata_df = pd.DataFrame(metadata)
        metadata_csv_path = os.path.join(output_folder, 'msi_metadata.csv')
        metadata_df.to_csv(metadata_csv_path, index=False)
        print(f"Successfully created {len(metadata)} MSI images for {len(metadata_df['subject_id'].unique())} subjects.")
        print(f"Metadata saved to: {metadata_csv_path}")
    else:
        print("No MSI images were successfully created.")

# Helper function for MSI creation
def motion_silhouette_image(sequence_images):
    msi = np.zeros_like(sequence_images[0], dtype=np.float32)
    for i in range(1, len(sequence_images)):
        diff = cv2.absdiff(sequence_images[i], sequence_images[i-1])
        msi += diff
    msi = (msi / np.max(msi) * 255).astype(np.uint8)
    return msi

# Usage example
base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
output_folder = '/kaggle/working/'
camera_angle = ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180']
create_msi_dataset(base_folder, output_folder,num_subjects=124, camera_angle=camera_angle)

1: Finished processing subject: 057
2: Finished processing subject: 086
3: Finished processing subject: 121
4: Finished processing subject: 061
5: Finished processing subject: 048
6: Finished processing subject: 053
7: Finished processing subject: 051


KeyboardInterrupt: 

In [3]:
import os
import zipfile

def zip_dataset(folder_path, output_zip_name):
    with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)
    print(f"Dataset zipped and exported to: {output_zip_name}")

# Usage example
base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
output_folder = '/kaggle/working/'
zip_dataset(output_folder, zip_filename)

NameError: name 'zip_filename' is not defined

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import shutil
import zipfile

def motion_silhouette_image(sequence_images):
    if len(sequence_images) < 2:
        return None
    
    msi = np.zeros_like(sequence_images[0], dtype=np.float32)
    for i in range(1, len(sequence_images)):
        diff = cv2.absdiff(sequence_images[i], sequence_images[i-1])
        msi += diff
    
    msi = msi / np.max(msi)  # Normalize to [0, 1]
    msi = (msi * 255).astype(np.uint8)  # Convert to uint8
    return msi

def create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=None):
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
    
    msi_images_folder = os.path.join(output_folder, 'msi_images')
    if not os.path.exists(msi_images_folder):
        os.makedirs(msi_images_folder)
    
    metadata = []
    counter = 0
    
    subject_folders = [f for f in os.listdir(base_folder) if f.isdigit()]
    if num_subjects is not None:
        subject_folders = subject_folders[:num_subjects]
    
    for subject_folder in subject_folders:
        counter += 1
        subject_path = os.path.join(base_folder, subject_folder, subject_folder)
        if not os.path.isdir(subject_path):
            print(f"No inner subject folder found for: {subject_folder}")
            continue
        
        for nm_folder in ['nm-01', 'nm-02', 'nm-03', 'nm-04', 'nm-05', 'nm-06']:
            nm_folder_path = os.path.join(subject_path, nm_folder)
            if not os.path.exists(nm_folder_path):
                print(f"No {nm_folder} folder found for subject: {subject_folder}")
                continue

            for cam_folder in camera_angles:
                cam_path = os.path.join(nm_folder_path, cam_folder)
                if not os.path.isdir(cam_path):
                    continue
                
                image_files = sorted([f for f in os.listdir(cam_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
                num_images = len(image_files)
                
                if num_images == 0:
                    print(f"No images found in {cam_path}")
                    continue
                
                
                section_size = num_images // 1
                split_sections = [image_files[i * section_size:(i + 1) * section_size] for i in range(5)]
                
                for section_idx, section in enumerate(split_sections):
                    sequence_images = []
                    for file in section:
                        image_path = os.path.join(cam_path, file)
                        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                        if image is not None and image.size > 0:
                            sequence_images.append(image)
                        else:
                            print(f"Failed to load image or empty image: {image_path}")
                    
                    if sequence_images:
                        msi = motion_silhouette_image(sequence_images)
                        
                        if msi is not None and msi.size > 0:
                            if msi.shape != (240, 240):
                                try:
                                    msi = cv2.resize(msi, (240, 240))
                                except cv2.error:
                                    print(f"Failed to resize MSI from {msi.shape} to (240, 240) for {cam_path}")
                                    continue
                            
                            msi_filename = f"{subject_folder}_{nm_folder}_{cam_folder}_{section_idx}.png"
                            msi_path = os.path.join(msi_images_folder, msi_filename)
                            cv2.imwrite(msi_path, msi)
                            
                            metadata.append({
                                'image_filename': msi_filename,
                                'label': int(subject_folder),
                                'cam_angle': int(cam_folder),
                                'nm_sequence': nm_folder,
                                'section': section_idx
                            })
                        else:
                            print(f"Failed to create valid MSI for {cam_path}")
                        
        print(f"{counter}: Finished processing subject: {subject_folder}")
    
    if metadata:
        df = pd.DataFrame(metadata)
        csv_path = os.path.join(output_folder, 'msi_metadata.csv')
        df.to_csv(csv_path, index=False)
        print(f"Successfully created {len(df)} MSI images for {len(df['label'].unique())} subjects.")
        print(f"Metadata saved to: {csv_path}")
    else:
        print("No MSI images were successfully created.")

def zip_dataset(output_folder, zip_filename):
    print(f"Creating zip file: {zip_filename}")
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(output_folder):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, output_folder)
                zipf.write(file_path, arcname)
    print(f"Zip file created: {zip_filename}")

# Usage example
base_folder = '/kaggle/input/casia-b/GaitDatasetB-silh'
output_folder = '/kaggle/working/'
zip_filename = '/kaggle/working/msi_dataset.zip'
camera_angles = ['000', '018', '036', '054', '072', '090', '108', '126', '144', '162', '180']

# Create the MSI dataset
create_dataset(base_folder, output_folder, num_subjects=None, camera_angles=camera_angles)

# Zip the dataset
zip_dataset(output_folder, zip_filename)

print(f"MSI dataset has been created and zipped. You can now download {zip_filename} from the Kaggle output.")

1: Finished processing subject: 057
2: Finished processing subject: 086
3: Finished processing subject: 121
4: Finished processing subject: 061
5: Finished processing subject: 048
6: Finished processing subject: 053
7: Finished processing subject: 051
8: Finished processing subject: 095
9: Finished processing subject: 018
10: Finished processing subject: 044
11: Finished processing subject: 016
12: Finished processing subject: 007
13: Finished processing subject: 009
14: Finished processing subject: 012
15: Finished processing subject: 029
16: Finished processing subject: 025
17: Finished processing subject: 078
18: Finished processing subject: 001
19: Finished processing subject: 056
20: Finished processing subject: 006
21: Finished processing subject: 120
22: Finished processing subject: 109
23: Finished processing subject: 042
24: Finished processing subject: 082
25: Finished processing subject: 055
26: Finished processing subject: 076
27: Finished processing subject: 091
28: Finish

/tmp/ipykernel_13/472859782.py:17: RuntimeWarning: invalid value encountered in divide
  msi = msi / np.max(msi)  # Normalize to [0, 1]
/tmp/ipykernel_13/472859782.py:18: RuntimeWarning: invalid value encountered in cast
  msi = (msi * 255).astype(np.uint8)  # Convert to uint8


114: Finished processing subject: 088
115: Finished processing subject: 116
116: Finished processing subject: 077
117: Finished processing subject: 028
118: Finished processing subject: 038
119: Finished processing subject: 118
120: Finished processing subject: 074
121: Finished processing subject: 079
122: Finished processing subject: 032
123: Finished processing subject: 030
124: Finished processing subject: 085
Successfully created 8154 MSI images for 124 subjects.
Metadata saved to: /kaggle/working/msi_metadata.csv
Creating zip file: /kaggle/working/msi_dataset.zip
Zip file created: /kaggle/working/msi_dataset.zip
MSI dataset has been created and zipped. You can now download /kaggle/working/msi_dataset.zip from the Kaggle output.
